# Теория вероятностей и математическая статистика 

## Иерархический кластерный анализ: количественные данные

*Алла Тамбовцева*

### Подготовка к работе

Импортируем необходимые библиотеки, модули и функции:

* библиотеку `pandas` для обработки данных;
* модуль `pyplot` из `matplotlib` и библиотеку `seaborn` для графики;
* функцию `StandardScaler()` для нормирования данных;
* функции `linkage()`, `dendrogram()`, `cut_tree()` для реализации иерархического кластерного анализа.

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, cut_tree

Загрузим данные из файла `flats_cian_upd.csv` – выгрузка с ЦИАН по квартирам в Москве:

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/allatambov/StatCS26/refs/heads/main/flats_cian_upd.csv")
print(df.shape)

In [ ]:
df.head()

В датафрейме 10054 строк и 14 столбцов. Переменные в файле:

* `price`: цена в рублях;
* `lprice`: логарифм цены;
* `square`: площадь квартиры, в кв. метрах;
* `rooms`: число комнат;
* `floor`: этаж;
* `mfloor`: число этажей в доме;
* `station`: станция метро;
* `metro`: доступность и расстояние до метро;
* `ametro`: шаговая доступность метро (1 – да, 0 – нет);
* `dmetro`: расстояние до метро (на транспорте или пешком), в минутах;
* `link`: ссылка на объявление;
* `add`: адрес;
* `lat`: широта;
* `lon`: долгота.

Посмотрим на описательные статистики:

In [ ]:
df.describe()

Судя по описательным статистикам, в данных есть нехарактерные значения – нетипично большие и дорогие квартиры. Проверим, что такие значения – не результат ошибки:

In [ ]:
# 5-комнатная квартира рядом с Площадью Революции
df[df["price"] == df["price"].max()]

In [ ]:
# 4-комнатная квартира площади 779 м^2 на Живописной улице
df[df["square"] == df["square"].max()]

Удалять такие наблюдения не будем, как раз большое и дорогое жилье выделится в отдельный кластер «элитного жилья». Однако чтобы не тратить много времени на запуск кластеризации более 10000 точек, случайным образом выберем 30 квартир в окрестности каждой станции метро:

In [ ]:
# таблица частот
tab = df["station"].value_counts()
tab

In [ ]:
# выбираем те станции, где квартир более 30
# забираем их названия – index

stations = tab[tab > 30].index
print(stations)

In [ ]:
# отбираем в chosen те квартиры, которые входят
# в список stations

chosen = df[df["station"].isin(stations)]

In [ ]:
# группируем квартиры по станции метро,
# извлекаем случайные выборки объема 30 из каждой группы
# random_state = 1234 – для воспроизводимости, чтобы
# у всех были одинаковые результаты

flats = chosen.groupby("station").sample(30, random_state = 1234)

Получили выборку из 2640 квартир:

In [ ]:
print(flats.shape)
flats.head()

### Нормирование данных

Выберем столбцы, на основе которых мы будет кластеризовать квартиры:

* логарифм цены;
* площадь квартиры;
* число комнат;
* этаж;
* расстояние до метро.

In [ ]:
small = flats[["lprice", "square", "rooms", "floor", "dmetro"]]
small.head()

В качестве способа шкалирования данных выберем *стандартизацию* – вычитание среднего и деление на стандартное отклонение, поскольку все показатели в количественной шкале (даже в абсолютной шкале). Создадим объект класса `StandardScaler()`, зарезервируем место под результаты, а затем применим метод `fit_transform()`:

* часть `fit` отвечает за оценку параметров, в данном случае расчет среднего и стандартного отклонения для каждого столбца;
* часть `transform` отвечает за преобразование, в данном случае за вычитание среднего и деление на стандартное отклонение.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(small)
X

В `X` сохранен массив стандартизованных значений (тип *датафрейм* теряем, но некритично, это те же пять столбцов с данными для Python). Всё готово к работе!

### Иерархический кластерный анализ

Попробуем реализовать иерархический кластерный анализ с настройками по-умолчанию – в `linkage()` будет использовано евклидово расстояние и метод ближнего соседа (метод одиночной связи): 

In [ ]:
# method = 'single'
# реализуем и сразу строим дендрограмму размера 16 на 9 дюймов

hc = linkage(X)

plt.figure(figsize = (16, 9))
dendrogram(hc);

Очевидно, что идея использовать такие параметры кластеризации – плохая. Метод ближнего соседа отличается тем, что он склонен образовывать монокластеры – кластеры из одного наблюдения. Здесь так и вышло, у дерева много висячих вершин, отсюда отсутствие более крупных групп, Python даже цветом рекомендует выделить один огромный кластер из всех квартир :)

Изменим метод агрегирования на метод дальнего соседа (полной связи):

In [ ]:
hc = linkage(X, method = "complete")

plt.figure(figsize = (16, 9))
dendrogram(hc);

Уже получше, но один из кластеров получился уж слишком большим, «растянутым». Понятно, что в него будут попадать очень разнообразные квартиры, а мы хотим получить более однородные кластеры. Попробуем самый эффективный метод – метод Варда (Уорда):

In [ ]:
hc = linkage(X, method = "ward")

plt.figure(figsize = (16, 9))
dendrogram(hc);

Вот теперь это похоже на что-то разумное. Выделим три кластера – «разрежем» дендрограмму примерно по расстоянию 55:

In [ ]:
# разрезаем дендрограмму на три ветки-группы
cut_tree(hc, n_clusters = 3)

Результат выше – метки кластеров, номера групп от 0 до 2 включительно. Однако они хранятся внутри вложенного массива, то есть с «лишними» квадратными скобками внутри. Уберем это ненужное измерение, применив метод `.reshape()`:

In [ ]:
# убираем одно измерение и получаем обычный массив как список
clusters_ = cut_tree(hc, n_clusters = 3).reshape(-1, )
clusters_

**Пояснение.** Посмотрим на размерность исходного массива:

In [ ]:
print(cut_tree(hc, n_clusters = 3).shape)

Число 1 здесь можно интерпретировать как число списков в каждой строке. Мы хотим от этих списков избавиться, чтобы получить обычный перечень значений, поэтому в `.reshape()` указываем значение `-1`. В итоге получается массив размерности 2640 на 0, что в рамках NumPy соответствует одномерным массивам, похожим на обычные списки в Python.

Добавим в датафрейм `flats` столбец `cluster_c` с полученными номерами кластеров, только прежде сделаем эти метки текстовыми. Никто не запрещает нумеровать кластеры целыми числами, но далее мы будем строить графики, и если мы будем раскрашивать точки, соответствующие разным кластерам, разными цветами, в случае числового номера Python будет делать градиентную растяжку от бледного цвета к яркому, что здесь неуместно.

In [ ]:
# превращаем integer в string
flats["cluster_c"] = clusters_.astype(str)
flats.head(10)

### Анализ результатов

Сгруппируем квартиры по полученным меткам кластеров `cluster_c` и посмотрим на описательные статистики ключевых показателей по каждой группе:

In [ ]:
# группируем по cluster_c
# выбираем столбец price/square/floor
# запрашиваем в agg характеристики

print(flats.groupby("cluster_c")["price"].agg(["count", "mean", "min", "max"]))
print(flats.groupby("cluster_c")["square"].agg(["count", "mean", "min", "max"]))
print(flats.groupby("cluster_c")["floor"].agg(["count", "mean", "min", "max"]))

Итак, в первом кластере с номером 0 больше всего квартир, это не самые дорогие квартиры с разнообразными значениями площади (от 10.7 до 593.3 кв. метров) и находящиеся в довольно разных домах (в среднем, квартиры на 10 этаже, но есть и те, что в домах не менее 69 этажей). 

Второй кластер с номером 1 самый маленький. Это, как и ожидалось, элитная недвижимость с очень высокой стоимостью, площадью не менее 74.4 кв. метров. 

Третий кластер с номером 2 – условно «среднестатистические» квартиры, достаточно многочисленные, со средней площадью 55 кв. метров (при этом площадь не выше 109.7 кв. метров) и с самой низкой средней стоимостью 25 миллионов рублей.

Визуализируем полученные результаты – построим ящики с усами:

In [ ]:
sns.boxplot(x = "price", y = "cluster_c", data = flats);

In [ ]:
sns.boxplot(x = "square", y = "cluster_c", data = flats);

О чем нам говорят полученные графики? Во-первых, распределение цены и площади в каждом кластере скошено вправо, несимметрично. А значит, если мы захотим запустить формальные тесты для проверки различий в группах, нам понадобятся непараметрические варианты, не предполагающие нормального распределения. Так, показать, что распределения цены и площади неодинаковы, мы сможем с помощью [критерия](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html) Краскела-Уоллиса вместо классической ANOVA для сравнения средних. 

Во-вторых, в каждой группе есть выбросы, и их достаточно много. Это можно интерпретировать как сигнал о том, что, возможно, выбранное нами число кластеров мало, внутри каждой группы можно выделить как минимум еще две – типичные и нетипичные значения. Если в наши задачи входило получение довольно общей классификации, трех кластеров достаточно, но вообще можно попробовать выделить 5-6 кластеров, чтобы нехарактерные наблюдения были сформированы в отдельные более маленькие группы. Если вернемся к дендрограмме, заметим, что при «разрезе» по расстоянию примерно 35 получим как раз 5 групп.

### Кластеры и география

Установим и импортируем библиотеку geopandas, это надстройка над `pandas`, которая позволяет загружать файлы с географической информацией (в частности, файлы `.geojson`) и отрисовывать карты.

In [ ]:
#!pip install geopandas

In [ ]:
import geopandas as gpd

В файле `Москва_Moscow.geojson` хранится географическая информация о районах Москвы – набор точек для определения и отрисовки границ района ([источник](https://github.com/timurkanaz/Russia_geojson_OSM/tree/master/GeoJson's) данных, там есть федеральные округа с делением на регионы и регионы с делением на районы, при открытии geojson на Github все отображается в виде готовой карты).

При загрузке geojson-файла через функцию `read_file()` данные внешне ничем не отличаются от обычного датафрейма `pandas`:

In [ ]:
df_geo = gpd.read_file("https://raw.githubusercontent.com/allatambov/LinStat25/refs/heads/main/%D0%9C%D0%BE%D1%81%D0%BA%D0%B2%D0%B0_Moscow.geojson")
df_geo.head()

Каждый район – это отдельный объект на карте. Чисто геометрически, этот объект – многоугольник, то есть какая-то область, ограниченная замкнутой ломаной линией, *polygon* на английском языке. Поэтому здесь в таблице в столбце `geometry` хранятся объекты специального типа *POLYGON*, которые внутри похожи на кортежи с парными координатами точек (широта и долгота) в выбранной географической проекции. По этим точкам район отрисовывается на карте. В столбце `geometry` также есть объекты типа *MULTIPOLYGON* для больших районов или районов со сложными границами, которые удобнее собрать из нескольких многоугольников.

К такому более продвинутому датафрейму типа *GeoDataFrame* можно применить метод `.plot()` и построить карту!

In [ ]:
df_geo.plot();

Или метод `boundary.plot()` для отрисовки только границ без заливки:

In [ ]:
df_geo.boundary.plot();

Так как в данных, которые мы отобрали для кластерного анализа, нет районов Зеленограда и некоторых районов Новой Москвы (мы основывались на станциях метро, а с метро там не очень), уберем их с карты:

In [ ]:
# список районов для удаления с карты

out = ["Троицкий административный округ", "район Кунцево", "район Силино", 
       "район Старое Крюково", "район Крюково",
       "район Матушкино", "район Савелки"]

# через ~ строим отрицание к результату isin()
# выбираем все, что не в out по столбцу district

df_geo = df_geo[~df_geo["district"].isin(out)]

Обновим карту:

In [ ]:
df_geo.boundary.plot();

Осталось нанести точки, соответствующие квартирам (у нас для них есть координаты – в столбцах `lon` и `lat`), на полученную карту. Для этого нужно в одних и тех же осях построить два графика – карту и диаграмму рассеивания с точками-квартирами:

In [ ]:
# создаем большой график 50 на 50 дюймов
# в fig остается изображение, которое выгрузим в файл, 
# в axes хранятся оси для редактирования


fig, axes = plt.subplots(figsize = (50, 50))

# строим карту в осях axes
df_geo.boundary.plot(ax = axes);

# в тех же осях axes строим диаграмму рассеивания
sns.scatterplot(data = flats, x = "lon", y = "lat", 
                hue = "cluster_c", s = 120, ax = axes);

**Пояснения.** Долготу `lon` (*longitude*) отмечаем по горизонтальной оси, широту `lat` (*latitude*) – по вертикальной. Цвет точек `hue` зависит от принадлежности к кластеру – метки в столбце `cluster_c`,  размер точек `s` увеличиваем до 120. 

Итого: хотя мы и взяли не очень большую выборку квартир, заметно, что дорогие квартиры из второго кластера (номер 1, оранжевые точки) располагаются, за редким исключением, в центре Москвы, причем в пределах Бульварного кольца (плюс район Хамовники), «среднестатистические» квартиры из третьего кластера (номер 2, зеленые точки) чаще встречаются на юге и юго-востоке Москвы и в Новой Москве.

**Дополнение (не запускайте, если переживаете за компьютер).** Кластеризуем все квартиры из исходного файла:

In [ ]:
# нормируем
X_full = scaler.fit_transform(df[["lprice", "square", "rooms", "floor", "dmetro"]])

# запускаем кластеризацию (примерно 5 минут)
hc = linkage(X_full, method = "ward")
plt.figure(figsize = (16, 9))
dendrogram(hc);

In [ ]:
# сохраняем метки кластеров
df["cluster"] = cut_tree(hc, n_clusters = 3).reshape(-1, ).astype(str)

In [ ]:
# наносим все на карту

fig, axes = plt.subplots(figsize = (50, 50))
df_geo.boundary.plot(ax = axes);
sns.scatterplot(data = df , x = "lon", y = "lat", hue = "cluster", ax = axes);

Цвета точек изменились, но точки в пределах кольца вместе с Хамовниками по-прежнему выделяются. Можем сохранить карту в файл:

In [ ]:
fig.savefig("map.pdf")